# Earnings Overnight Backtest

Options position opened on the trading day **before** earnings (T-1, last 30 min
before close) and closed on **earnings day** (T, at the open) — held through the
announcement. Configurable via `structure`:

- **`"single"`** — one 2% OTM leg, call or put, whichever has the higher
  OI × Volume ("follow smart money"). A **directional** bet.
- **`"strangle"`** — both 2% OTM legs (宽跨式). **Non-directional**: profits on
  a large move in either direction.
- **`"straddle"`** — both ATM legs (跨式). Most premium and gamma.

Cost capped at $1,000 per position (the **combined** debit for two-leg
structures); events whose candidate busts the cap are dropped.

**Scope**: this strategy targets **individual stocks** (AAPL, MSFT, NVDA, etc.).
ETFs like SPY / QQQ / IWM don't file earnings in the EODHD feed and have
empty earnings caches — running this on an ETF will raise
`ValueError("Empty earnings calendar")`.

**Prerequisites** — pre-populate the data caches via the CLI:

```bash
# Options + stock for the symbol (EODHD paid plan required)
just download AAPL

# Earnings calendar (this package's own CLI)
just download-earnings --symbols AAPL
```

In [1]:
from dotenv import load_dotenv

load_dotenv()

from datetime import date

from options_strategies.earnings_overnight import (
    EarningsOvernightConfig,
    run_earnings_overnight,
)
from options_strategies.shared import load_earnings_overnight_data
from backtest_charts import BacktestReport

## Configuration

`structure` picks the position shape:

| value | legs | bet |
|---|---|---|
| `"single"` | 2% OTM call **or** put (higher OI×Vol) | **directional** — the OI ranking picks the side |
| `"strangle"` | 2% OTM call **and** put | **non-directional** — needs a big move either way |
| `"straddle"` | ATM call **and** put | non-directional, most premium & gamma |

Two-leg structures cost ~2× a single leg, and `cost_cap_usd` applies to the
**combined** debit — `"straddle"` frequently busts a $1,000 cap on
higher-priced underlyings. Two-leg events are also all-or-nothing: if either
leg fails the liquidity filters the whole event is skipped.

`as_of_date` defaults to `date.today()` so EODHD's pre-published future
events are filtered out (no peeking ahead in a point-in-time backtest).
Override to a specific past date to inspect what the strategy "knew"
at that point in time.

In [2]:
config = EarningsOvernightConfig(
    symbol="AAPL",  # individual stock, NOT an ETF
    capital=100_000.0,
    # Position shape: "single" | "strangle" | "straddle"
    structure="strangle",
    # Strike selection
    otm_target_pct=0.02,        # 2% OTM (ignored when structure="straddle")
    otm_tolerance_pct=0.005,    # ±0.5% strike band
    reference_price="high",     # T-1 daily high ≈ 3:30 PM
    # Liquidity filter ("follow smart money")
    min_oi=100,
    min_volume=50,
    # Expiration
    max_entry_dte=14,           # "nearest expiry" — keep it cheap
    min_entry_dte=0,
    # Cost cap — for two-leg structures this is the COMBINED debit
    cost_cap_usd=1_000.0,
    # Capital split between the call and put legs
    call_weight=0.5,
    put_weight=0.5,
    # Point-in-time cutoff
    as_of_date=date.today(),
)
config

EarningsOvernightConfig(symbol='AAPL', capital=100000.0, quantity=1, multiplier=100, max_positions=1, start_date=None, end_date=None, as_of_date=datetime.date(2026, 8, 6), structure='strangle', otm_target_pct=0.02, otm_tolerance_pct=0.005, reference_price='high', min_oi=100, min_volume=50, max_entry_dte=14, min_entry_dte=0, exit_dte=1, exit_dte_tolerance=1, cost_cap_usd=1000.0, call_weight=0.5, put_weight=0.5)

## Load Data

In [3]:
print(f"Loading data for {config.symbol}…")
options, stock, earnings = load_earnings_overnight_data(
    config.symbol,
    as_of_date=config.as_of_date,
)
print(f"  Options: {len(options):,} rows")
print(f"  Stock:   {len(stock):,} rows")
print(f"  Earnings: {sum(len(v) for v in earnings.values())} events cached")
for code_, dates in earnings.items():
    print(f"    {code_}: {len(dates)} events ({dates[0].date()} → {dates[-1].date()})")

Loading data for AAPL…
  Options: 748,428 rows
  Stock:   11,503 rows
  Earnings: 11 events cached
    AAPL: 11 events (2024-02-01 → 2026-07-30)


## Run Backtest

In [4]:
print("Running earnings-overnight backtest…")
result = run_earnings_overnight(options, stock, earnings, config)

Running earnings-overnight backtest…


e:\TMP\trade-system\.venv\Lib\site-packages\empyrical\stats.py:1424: RuntimeWarning: invalid value encountered in scalar divide
  return np.abs(np.percentile(returns, 95)) / np.abs(np.percentile(returns, 5))
e:\TMP\trade-system\.venv\Lib\site-packages\empyrical\stats.py:1424: RuntimeWarning: invalid value encountered in scalar divide
  return np.abs(np.percentile(returns, 95)) / np.abs(np.percentile(returns, 5))
e:\TMP\trade-system\.venv\Lib\site-packages\empyrical\stats.py:1424: RuntimeWarning: invalid value encountered in scalar divide
  return np.abs(np.percentile(returns, 95)) / np.abs(np.percentile(returns, 5))


## Summary

In [5]:
s = result.summary
print("═══ Earnings Overnight Portfolio Summary ═══")
print(f"  Total trades:    {s.get('total_trades', 0)}")
print(f"  Win rate:        {s.get('win_rate', 0):.1%}")
print(f"  Total P&L:       ${s.get('total_pnl', 0):,.2f}")
print(f"  Max drawdown:    {s.get('max_drawdown', 0):.2%}")
print(f"  Sharpe ratio:    {s.get('sharpe_ratio', 0):.2f}")
print(f"  Profit factor:   {s.get('profit_factor', 0):.2f}")
print(f"  Avg days held:   {s.get('avg_days_in_trade', 0):.1f}")

═══ Earnings Overnight Portfolio Summary ═══
  Total trades:    15
  Win rate:        60.0%
  Total P&L:       $361.50
  Max drawdown:    -0.12%
  Sharpe ratio:    0.70
  Profit factor:   1.90
  Avg days held:   1.0


## Per-Leg Results

In [6]:
for name, leg in result.leg_results.items():
    ls = leg.summary
    print(f"\n  ── {name} leg ──")
    print(
        f"    Trades: {ls.get('total_trades', 0)}  "
        f"Win rate: {ls.get('win_rate', 0):.1%}  "
        f"P&L: ${ls.get('total_pnl', 0):,.2f}"
    )


  ── earnings_call leg ──
    Trades: 7  Win rate: 57.1%  P&L: $-42.00

  ── earnings_put leg ──
    Trades: 8  Win rate: 62.5%  P&L: $403.50


## Trade Log

In [7]:
if not result.trade_log.empty:
    result.trade_log.head(20)
else:
    print("No trades — check filters / cache coverage.")

## Caveats (read these before trusting the numbers)

1. **T-1 daily `high` is a 3:30 PM proxy.** optopsy's data is EOD-only;
   the actual 3:30 PM price could be lower than the daily high. For
   SPY/QQQ the 3:00–4:00 PM range is typically < 0.1 %, so the proxy
   is tight; for names with volatile closes the strike ends up further
   OTM than intended.
2. **T EOD is a 9:30 AM open proxy.** The post-earnings exit P&L uses
   the closing bid/ask of the day earnings are announced, which is
   hours after the open. The realized volatility of an actual
   9:30 AM open → 4 PM close trade is understated here.
3. **$1,000 cost cap is strict.** If the top OI×Volume candidate's
   ask × 100 × 1 exceeds the cap, the event is dropped (no entry, no
   quantity rescaling, no fallback to the lower-score side).
4. **Single-symbol workflow.** Looping over a portfolio of symbols is
   a future extension.
5. **EODHD options history is ~2 years.** SPY/QQQ have enough
   coverage; IWM (~1.4 years) is too thin for a 2-year backtest.


## Visualizations

### Create report

In [8]:
report = BacktestReport(result, capital=config.capital)

### Equity curve

In [ ]:
report.plot_equity()

### Cumulative P&L by leg

In [ ]:
report.plot_cum_pnl()

### Per-trade P&L distribution

In [ ]:
report.plot_pnl_dist()

### Exit type breakdown

In [ ]:
report.plot_exits()

### Strategy summary table

In [ ]:
report.plot_summary()

### Full dashboard

In [ ]:
report.plot_dashboard()